In [1]:
# ============================================================
# PHASE 6 — Cell 1: Setup + Load Original Split
# ============================================================
!pip install wfdb -q

from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import wfdb
from scipy.signal import butter, filtfilt
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'mitdb')
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_DIR, 'splits')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')

print(f"wfdb: {wfdb.__version__}")

# Load original split
with open(os.path.join(SPLITS_DIR, 'mitbih_patient_split.json')) as f:
    split_orig = json.load(f)

train_records_orig = sorted(split_orig['train'])
val_records = sorted(split_orig['val'])
test_records_orig = sorted(split_orig['test'])

print(f"\n✅ Original split loaded")
print(f"   Train: {len(train_records_orig)} records")
print(f"   Val:   {len(val_records)} records")
print(f"   Test:  {len(test_records_orig)} records")

# Verify 201 and 202 are in different splits
print(f"\n🔍 201/202 location check:")
print(f"   201 in train: {'201' in train_records_orig}")
print(f"   201 in test:  {'201' in test_records_orig}")
print(f"   202 in train: {'202' in train_records_orig}")
print(f"   202 in test:  {'202' in test_records_orig}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 961.7 kB/s eta 0:00:00
Mounted at /content/drive
wfdb: 4.3.1

✅ Original split loaded
   Train: 20 records
   Val:   5 records
   Test:  23 records

🔍 201/202 location check:
   201 in train: True
   201 in test:  False
   202 in train: False
   202 in test:  True


In [2]:
# ============================================================
# PHASE 6 — Cell 2: Create Strict Subject-Disjoint Split
# ============================================================
# MIT-BIH: 201 and 202 are from the SAME SUBJECT (S025)
# Standard DS1/DS2: 201 → train, 202 → test (LEAKAGE)
# Strict version: BOTH in train (zero leakage)

# Move 202 from test to train
test_records_strict = sorted([r for r in test_records_orig if r != '202'])
train_records_strict = sorted(train_records_orig + ['202'])

print("=" * 70)
print("STRICT SUBJECT-DISJOINT SPLIT")
print("=" * 70)

print(f"\nTrain: {len(train_records_strict)} records")
print(f"   Added: 202 (moved from test)")
print(f"   Total: {train_records_strict}")

print(f"\nVal: {len(val_records)} records")
print(f"   {val_records}")

print(f"\nTest: {len(test_records_strict)} records")
print(f"   Removed: 202")
print(f"   Total: {test_records_strict}")

# Verify no leakage
print(f"\n🔍 Leakage check:")
print(f"   201 in train: {'201' in train_records_strict}")
print(f"   202 in train: {'202' in train_records_strict}")
print(f"   201 in test:  {'201' in test_records_strict}")
print(f"   202 in test:  {'202' in test_records_strict}")
print(f"   ✅ Both 201 and 202 in SAME split (train)")

# Save strict split
split_strict = {
    'protocol': 'STRICT_SUBJECT_DISJOINT (202 moved to train)',
    'random_seed': 42,
    'notes': '201/202 both in train (same subject S025)',
    'train_records': train_records_strict,
    'val_records': val_records,
    'test_records': test_records_strict,
    'moved_for_strict_split': ['202']
}

split_path = os.path.join(SPLITS_DIR, 'mitbih_strict_split_v4.json')
with open(split_path, 'w') as f:
    json.dump(split_strict, f, indent=2)

print(f"\n✅ Strict split saved: {split_path}")

STRICT SUBJECT-DISJOINT SPLIT

Train: 21 records
   Added: 202 (moved from test)
   Total: ['101', '102', '104', '106', '108', '109', '112', '114', '115', '116', '118', '119', '124', '201', '202', '203', '205', '207', '215', '220', '223']

Val: 5 records
   ['107', '122', '208', '209', '230']

Test: 22 records
   Removed: 202
   Total: ['100', '103', '105', '111', '113', '117', '121', '123', '200', '210', '212', '213', '214', '217', '219', '221', '222', '228', '231', '232', '233', '234']

🔍 Leakage check:
   201 in train: True
   202 in train: True
   201 in test:  False
   202 in test:  False
   ✅ Both 201 and 202 in SAME split (train)

✅ Strict split saved: /content/drive/MyDrive/ecg-transcovnet/splits/mitbih_strict_split_v4.json


In [3]:
# ============================================================
# PHASE 6 — Cell 3: Step 2 Rebuild for Strict Split
# ============================================================
AAMI_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F',
    '/': 'Q', 'f': 'Q', 'Q': 'Q'
}
ORIG_TO_IDX = {'N': 0, 'S': 1, 'V': 2, 'F': 3, 'Q': 4}
LABEL_MAP_4C = {0: 0, 1: 1, 2: 2, 4: 3}  # 4-class (N, S, V, Q)
CLASS_NAMES = ['N', 'S', 'V', 'Q']

FS = 360
PRE_SAMPLES = 99
POST_SAMPLES = 160
BEAT_LEN = PRE_SAMPLES + POST_SAMPLES


def bandpass_filter(signal, fs=FS, low=0.5, high=40.0, order=3):
    nyq = fs / 2.0
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)


def extract_beats_with_rr(record_id, raw_dir=RAW_DIR):
    """Extract beats + 4 RR features + 4-class labels."""
    sig, fields = wfdb.rdsamp(os.path.join(raw_dir, record_id))
    ann = wfdb.rdann(os.path.join(raw_dir, record_id), 'atr')

    lead = sig[:, 0]
    filtered = bandpass_filter(lead)

    r_peaks, orig_classes = [], []
    for sample, sym in zip(ann.sample, ann.symbol):
        cls = AAMI_MAP.get(sym)
        if cls is None:
            continue
        r_peaks.append(sample)
        orig_classes.append(ORIG_TO_IDX[cls])

    r_peaks = np.array(r_peaks)
    orig_classes = np.array(orig_classes)
    n = len(filtered)
    rr_intervals = np.diff(r_peaks)

    X, y, R = [], [], []
    for i, (sample, orig_cls) in enumerate(zip(r_peaks, orig_classes)):
        if orig_cls == 3:  # Drop F
            continue

        new_cls = LABEL_MAP_4C[orig_cls]

        start = sample - PRE_SAMPLES
        end = sample + POST_SAMPLES
        if start < 0 or end > n:
            continue

        beat = filtered[start:end]

        pre_rr = rr_intervals[i-1] if i > 0 else (rr_intervals[0] if len(rr_intervals) > 0 else FS)
        post_rr = rr_intervals[i] if i < len(rr_intervals) else (rr_intervals[-1] if len(rr_intervals) > 0 else FS)
        rr_ratio = pre_rr / (post_rr + 1e-8)
        local_hr = 60.0 * FS / ((pre_rr + post_rr) / 2.0 + 1e-8)

        X.append(beat)
        y.append(new_cls)
        R.append([pre_rr, post_rr, rr_ratio, local_hr])

    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.int64),
            np.array(R, dtype=np.float32))


def build_split_arrays(record_list, label):
    Xs, ys, Rs, rec_ids = [], [], [], []
    for rec in record_list:
        X, y, R = extract_beats_with_rr(rec)
        Xs.append(X); ys.append(y); Rs.append(R)
        rec_ids.extend([rec] * len(y))
        print(f'  {rec}: {len(y)} beats')
    X_all = np.concatenate(Xs, axis=0)
    y_all = np.concatenate(ys, axis=0)
    R_all = np.concatenate(Rs, axis=0)
    print(f'{label} TOTAL: X={X_all.shape}, R={R_all.shape}')
    print(f'  Labels: {dict(Counter(y_all))}\n')
    return X_all, y_all, R_all, np.array(rec_ids)


print("Extracting TRAIN (strict, includes 202)...")
X_train, y_train, R_train, rec_train = build_split_arrays(train_records_strict, 'TRAIN')

print("Extracting VAL...")
X_val, y_val, R_val, rec_val = build_split_arrays(val_records, 'VAL')

print("Extracting TEST (strict, excludes 202)...")
X_test, y_test, R_test, rec_test = build_split_arrays(test_records_strict, 'TEST')

Extracting TRAIN (strict, includes 202)...
  101: 1864 beats
  102: 2186 beats
  104: 2227 beats
  106: 2027 beats
  108: 1760 beats
  109: 2529 beats
  112: 2538 beats
  114: 1875 beats
  115: 1952 beats
  116: 2411 beats
  118: 2277 beats
  119: 1987 beats
  124: 1613 beats
  201: 1961 beats
  202: 2134 beats
  203: 2979 beats
  205: 2644 beats
  207: 1859 beats
  215: 3361 beats
  220: 2046 beats
  223: 2590 beats
TRAIN TOTAL: X=(46820, 259), R=(46820, 4)
  Labels: {np.int64(0): 39236, np.int64(3): 4151, np.int64(1): 614, np.int64(2): 2819}

Extracting VAL...
  107: 2136 beats
  122: 2474 beats
  208: 2581 beats
  209: 3004 beats
  230: 2255 beats
VAL TOTAL: X=(12450, 259), R=(12450, 4)
  Labels: {np.int64(3): 2079, np.int64(2): 1053, np.int64(0): 8933, np.int64(1): 385}

Extracting TEST (strict, excludes 202)...
  100: 2271 beats
  103: 2083 beats
  105: 2572 beats
  111: 2124 beats
  113: 1794 beats
  117: 1534 beats
  121: 1862 beats
  123: 1517 beats
  200: 2598 beats
  210: 263

In [4]:
# ============================================================
# PHASE 6 — Cell 4: Normalize + Save Strict Split Data
# ============================================================
def per_beat_normalize(X, eps=1e-8):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    return (X - m) / (s + eps)


X_train_n = per_beat_normalize(X_train)
X_val_n = per_beat_normalize(X_val)
X_test_n = per_beat_normalize(X_test)

# Save strict data
np.savez_compressed(os.path.join(PROCESSED_DIR, 'train_v4_strict.npz'),
                    X=X_train_n, y=y_train, R=R_train, record_id=rec_train)
np.savez_compressed(os.path.join(PROCESSED_DIR, 'val_v4_strict.npz'),
                    X=X_val_n, y=y_val, R=R_val, record_id=rec_val)
np.savez_compressed(os.path.join(PROCESSED_DIR, 'test_v4_strict.npz'),
                    X=X_test_n, y=y_test, R=R_test, record_id=rec_test)

# Save config
with open(os.path.join(PROCESSED_DIR, 'preprocessing_config_v4_strict.json'), 'w') as f:
    json.dump({
        'split_file': 'mitbih_strict_split_v4.json',
        'protocol': 'STRICT_SUBJECT_DISJOINT',
        'label_protocol': '4-class (N, S, V, Q)',
        'class_names': CLASS_NAMES,
        'pre_samples': PRE_SAMPLES,
        'post_samples': POST_SAMPLES,
        'beat_len': BEAT_LEN,
        'fs': FS,
        'train_beats': int(len(y_train)),
        'val_beats': int(len(y_val)),
        'test_beats': int(len(y_test)),
        'notes': '202 moved from test to train (strict subject-disjoint)'
    }, f, indent=2)

print("✅ Strict split data saved:")
print(f"   train_v4_strict.npz: X={X_train_n.shape}, R={R_train.shape}")
print(f"   val_v4_strict.npz:   X={X_val_n.shape}, R={R_val.shape}")
print(f"   test_v4_strict.npz:  X={X_test_n.shape}, R={R_test.shape}")

# Verify
print("\n" + "=" * 70)
print("VERIFICATION")
print("=" * 70)
for f in ['train_v4_strict.npz', 'val_v4_strict.npz', 'test_v4_strict.npz', 'preprocessing_config_v4_strict.json']:
    path = os.path.join(PROCESSED_DIR, f)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"✅ {f}: {size:.2f} MB")
    else:
        print(f"❌ {f}: NOT FOUND")

✅ Strict split data saved:
   train_v4_strict.npz: X=(46820, 259), R=(46820, 4)
   val_v4_strict.npz:   X=(12450, 259), R=(12450, 4)
   test_v4_strict.npz:  X=(49377, 259), R=(49377, 4)

VERIFICATION
✅ train_v4_strict.npz: 43.06 MB
✅ val_v4_strict.npz: 11.42 MB
✅ test_v4_strict.npz: 45.57 MB
✅ preprocessing_config_v4_strict.json: 0.00 MB


In [5]:
# ============================================================
# PHASE 6 — Cell 5: Train ECG-TransCovNet (strict split, no re-tuning)
# ============================================================
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")


def build_ecg_transcovnet(beat_len=259, n_rr_features=4, n_classes=4, seed=42):
    tf.keras.utils.set_random_seed(seed)
    reg = tf.keras.regularizers.l2(1e-4)

    # CNN branch
    wave_in = layers.Input(shape=(beat_len, 1), name='waveform')
    x_cnn = wave_in
    for i, (f, k) in enumerate(zip((64, 128, 256), (7, 5, 3))):
        x_cnn = layers.Conv1D(f, k, padding='same', kernel_regularizer=reg)(x_cnn)
        x_cnn = layers.BatchNormalization()(x_cnn)
        x_cnn = layers.Activation('relu')(x_cnn)
        x_cnn = layers.MaxPooling1D(2)(x_cnn)
        x_cnn = layers.Dropout(0.2)(x_cnn)
    cnn_feat = layers.GlobalAveragePooling1D()(x_cnn)

    # Transformer branch
    x_trans = layers.Dense(128, name='trans_proj')(wave_in)
    pos_embed = layers.Embedding(input_dim=beat_len, output_dim=128)(tf.range(beat_len))
    x_trans = x_trans + pos_embed
    for i in range(2):
        attn_out = layers.MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.1)(x_trans, x_trans)
        x_trans = layers.Add()([x_trans, attn_out])
        x_trans = layers.LayerNormalization(epsilon=1e-6)(x_trans)
        ffn_out = layers.Dense(512, activation='gelu', kernel_regularizer=reg)(x_trans)
        ffn_out = layers.Dropout(0.1)(ffn_out)
        ffn_out = layers.Dense(128, kernel_regularizer=reg)(ffn_out)
        x_trans = layers.Add()([x_trans, ffn_out])
        x_trans = layers.LayerNormalization(epsilon=1e-6)(x_trans)
    trans_feat = layers.GlobalAveragePooling1D()(x_trans)

    # Attention fusion
    concat = layers.Concatenate()([cnn_feat, trans_feat])
    alpha_logits = layers.Dense(2)(concat)
    alpha = layers.Softmax()(alpha_logits)
    alpha_cnn = layers.Lambda(lambda x: x[:, 0:1])(alpha)
    alpha_trans = layers.Lambda(lambda x: x[:, 1:2])(alpha)

    cnn_proj = layers.Dense(256)(cnn_feat)
    trans_proj = layers.Dense(256)(trans_feat)
    cnn_weighted = layers.Multiply()([alpha_cnn, cnn_proj])
    trans_weighted = layers.Multiply()([alpha_trans, trans_proj])
    fused = layers.Add()([cnn_weighted, trans_weighted])

    # RR branch
    rr_in = layers.Input(shape=(n_rr_features,), name='rr_features')
    r = layers.Dense(32, activation='relu', kernel_regularizer=reg)(rr_in)
    r = layers.BatchNormalization()(r)
    r = layers.Dense(16, activation='relu', kernel_regularizer=reg)(r)

    # Classifier
    merged = layers.Concatenate()([fused, r])
    d = layers.Dense(64, activation='relu', kernel_regularizer=reg)(merged)
    d = layers.Dropout(0.4)(d)
    out = layers.Dense(n_classes, activation='softmax')(d)

    model = Model(inputs=[wave_in, rr_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def add_channel(X):
    return X[..., np.newaxis].astype(np.float32)


def make_ds(X, R, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices(({'waveform': X, 'rr_features': R}, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    return ds.batch(128).prefetch(tf.data.AUTOTUNE)


# ============================================================
# Prepare data with SMOTE (same as main)
# ============================================================
print("\n" + "="*70)
print("PREPARING DATA WITH SMOTE")
print("="*70)

combined_train = np.concatenate([X_train_n, R_train], axis=1)
counts = Counter(y_train)
majority = max(counts.values())
strat = {}
for cls, cnt in counts.items():
    if cnt < majority:
        strat[cls] = max(min(int(majority * 0.25), majority), cnt)
min_cnt = min(counts.values())
k = min(5, max(1, min_cnt - 1))
sm = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=strat)
combined_res, y_train_res = sm.fit_resample(combined_train, y_train)

X_train_res = combined_res[:, :BEAT_LEN]
R_train_res = combined_res[:, BEAT_LEN:]

print(f"After SMOTE: {dict(Counter(y_train_res))}")

# Class weights (best config from main)
classes_present = np.unique(y_train_res)
base_w = compute_class_weight('balanced', classes=classes_present, y=y_train_res)
base_w_dict = dict(zip(classes_present.tolist(), base_w.tolist()))
best_config = {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8}
cw = {i: base_w_dict[i] * best_config[CLASS_NAMES[i]] for i in classes_present}
print(f"Class weights: {cw}")

# Datasets
train_ds = make_ds(add_channel(X_train_res), R_train_res.astype(np.float32),
                    y_train_res.astype(np.int64), training=True)
val_ds = make_ds(add_channel(X_val_n), R_val.astype(np.float32),
                  y_val.astype(np.int64))

# Train
print("\n" + "="*70)
print("TRAINING ECG-TransCovNet (strict split)")
print("="*70)

model = build_ecg_transcovnet(BEAT_LEN, seed=42)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=6, restore_best_weights=True, verbose=1
    )],
    class_weight=cw,
    verbose=1
)

print(f"\n✅ Training complete")
print(f"   Best val loss: {min(history.history['val_loss']):.4f}")
print(f"   Best val accuracy: {max(history.history['val_accuracy']):.4f}")

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

PREPARING DATA WITH SMOTE
After SMOTE: {np.int64(0): 39236, np.int64(3): 9809, np.int64(1): 9809, np.int64(2): 9809}
Class weights: {np.int64(0): 0.4375, np.int64(1): 2.625, np.int64(2): 1.4000000000000001, np.int64(3): 1.4000000000000001}

TRAINING ECG-TransCovNet (strict split)
Epoch 1/30
537/537 ━━━━━━━━━━━━━━━━━━━━ 97s 123ms/step - accuracy: 0.8241 - loss: 0.5461 - val_accuracy: 0.4116 - val_loss: 1.3543
Epoch 2/30
537/537 ━━━━━━━━━━━━━━━━━━━━ 49s 90ms/step - accuracy: 0.9226 - loss: 0.2817 - val_accuracy: 0.4878 - val_loss: 1.0341
Epoch 3/30
537/537 ━━━━━━━━━━━━━━━━━━━━ 56s 104ms/step - accuracy: 0.9430 - loss: 0.2025 - val_accuracy: 0.7643 - val_loss: 1.0431
Epoch 4/30
537/537 ━━━━━━━━━━━━━━━━━━━━ 52s 97ms/step - accuracy: 0.9529 - loss: 0.1696 - val_accuracy: 0.5280 - val_loss: 1.8566
Epoch 5/30
537/537 ━━━━━━━━━━━━━━━━━━━━ 52s 97ms/step - accuracy: 0.9616 - loss: 0.1464 - val_accuracy: 0

In [6]:
# ============================================================
# PHASE 6 — Cell 6: Locked Test Evaluation (Strict)
# ============================================================
print("=" * 70)
print("🔒 STRICT SPLIT — LOCKED TEST EVALUATION")
print("=" * 70)

X_test_c = add_channel(X_test_n)
R_test_f = R_test.astype(np.float32)

y_pred_probs = model.predict({'waveform': X_test_c, 'rr_features': R_test_f}, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n" + "="*70)
print("STRICT SPLIT — TEST RESULTS")
print("="*70)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES,
                             digits=4, zero_division=0))

strict_accuracy = float((y_pred == y_test).mean())
strict_macro_f1 = float(f1_score(y_test, y_pred, average='macro'))
strict_weighted_f1 = float(f1_score(y_test, y_pred, average='weighted'))

print(f"Accuracy:    {strict_accuracy:.4f}")
print(f"Macro-F1:    {strict_macro_f1:.4f}")
print(f"Weighted-F1: {strict_weighted_f1:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"{'':>6}", "  ".join(f"{c:>6s}" for c in CLASS_NAMES))
for i, row in enumerate(cm):
    print(f"{CLASS_NAMES[i]:>6}", "  ".join(f"{v:6d}" for v in row))

# Save model + results
model.save(os.path.join(MODELS_DIR, 'ecg_transcovnet_strict_locked.keras'))

with open(os.path.join(OUTPUTS_DIR, 'phase6_strict_results.json'), 'w') as f:
    json.dump({
        'phase': 'Phase 6 — Supplementary Strict Subject-Disjoint',
        'split_protocol': 'STRICT_SUBJECT_DISJOINT (202 in train)',
        'model': 'ECG-TransCovNet',
        'config': best_config,
        'test_accuracy': strict_accuracy,
        'test_macro_f1': strict_macro_f1,
        'test_weighted_f1': strict_weighted_f1,
        'test_beats': int(len(y_test)),
        'train_beats': int(len(y_train)),
        'confusion_matrix': cm.tolist()
    }, f, indent=2)

print(f"\n✅ Strict model + results saved")

🔒 STRICT SPLIT — LOCKED TEST EVALUATION

STRICT SPLIT — TEST RESULTS
              precision    recall  f1-score   support

           N     0.9366    0.9320    0.9343     42423
           S     0.2223    0.1409    0.1724      1782
           V     0.6969    0.9700    0.8110      3363
           Q     0.2705    0.2023    0.2315      1809

    accuracy                         0.8793     49377
   macro avg     0.5316    0.5613    0.5373     49377
weighted avg     0.8701    0.8793    0.8727     49377

Accuracy:    0.8793
Macro-F1:    0.5373
Weighted-F1: 0.8727

Confusion Matrix:
            N       S       V       Q
     N  39538     836    1067     982
     S   1208     251     318       5
     V     59      42    3262       0
     Q   1409       0      34     366

✅ Strict model + results saved


In [7]:
import numpy as np
import os
from collections import Counter

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')

def load_split_v2(name):
    d = np.load(os.path.join(PROCESSED_DIR, f'{name}_v2.npz'))
    return d['X'], d['y'], d['R'], d['record_id']

def load_split_v1(name):
    d = np.load(os.path.join(PROCESSED_DIR, f'{name}.npz'))
    return d['X'], d['y'], d['R'], d['record_id']

print('='*70)
print('DIAGNOSIS: Where did the paced-beat (Q-source) records end up?')
print('='*70)

PACED_RECORDS = {'102', '104', '107', '217'}

for label, loader in [('STANDARD (v1)', load_split_v1), ('STRICT (v2)', loader:=load_split_v2)]:
    print(f'\n--- {label} split ---')
    for split_name in ['train', 'val', 'test']:
        _, y, _, rec = loader(split_name)
        rec_set = set(rec)
        paced_here = rec_set & PACED_RECORDS
        # Q label is 4 in the RAW (pre-remap) label scheme (before drop_F_and_remap converts it to 3)
        q_count_raw = int((y == 4).sum())
        print(f'  {split_name:5s}: {len(rec_set):2d} patients | paced records present: {sorted(paced_here) or "NONE"} | raw Q-beats: {q_count_raw}')

DIAGNOSIS: Where did the paced-beat (Q-source) records end up?

--- STANDARD (v1) split ---
  train: 20 patients | paced records present: ['102', '104'] | raw Q-beats: 4151
  val  :  5 patients | paced records present: ['107'] | raw Q-beats: 2079
  test : 23 patients | paced records present: ['217'] | raw Q-beats: 1809

--- STRICT (v2) split ---
  train: 37 patients | paced records present: ['107', '217'] | raw Q-beats: 0
  val  :  5 patients | paced records present: ['104'] | raw Q-beats: 0
  test :  6 patients | paced records present: ['102'] | raw Q-beats: 0


In [8]:
print('='*70)
print('What raw label value do known PACED (Q-source) records actually have in v2?')
print('='*70)

PACED_RECORDS = {'102', '104', '107', '217'}

for split_name in ['train', 'val', 'test']:
    _, y, _, rec = load_split_v2(split_name)
    print(f'\n--- {split_name}_v2 ---')
    print(f'  Overall label distribution: {Counter(y)}')
    for p in sorted(set(rec) & PACED_RECORDS):
        mask = rec == p
        print(f'  Record {p} (known paced/Q-source): label distribution = {Counter(y[mask])}')

What raw label value do known PACED (Q-source) records actually have in v2?

--- train_v2 ---
  Overall label distribution: Counter({np.int64(0): 75332, np.int64(1): 6018, np.int64(2): 3890})
  Record 107 (known paced/Q-source): label distribution = Counter({np.int64(2): 2077, np.int64(1): 59})
  Record 217 (known paced/Q-source): label distribution = Counter({np.int64(2): 1802, np.int64(0): 244, np.int64(1): 162})

--- val_v2 ---
  Overall label distribution: Counter({np.int64(0): 8415, np.int64(2): 2062, np.int64(1): 833})
  Record 104 (known paced/Q-source): label distribution = Counter({np.int64(2): 2062, np.int64(0): 163, np.int64(1): 2})

--- test_v2 ---
  Overall label distribution: Counter({np.int64(0): 9626, np.int64(2): 2087, np.int64(1): 384})
  Record 102 (known paced/Q-source): label distribution = Counter({np.int64(2): 2083, np.int64(0): 99, np.int64(1): 4})


In [9]:
import os, json, time
import numpy as np
import wfdb
from scipy.signal import butter, filtfilt

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw', 'mitdb')
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')

# ---- Standard split (our already-validated one) ----
DS1 = ['101','106','108','109','112','114','115','116','118','119',
       '122','124','201','203','205','207','208','209','215','220','223','230']
DS2 = ['100','103','105','111','113','117','121','123','200','202',
       '210','212','213','214','219','221','222','228','231','232','233','234']
PACED_RECORDS = ['102', '104', '107', '217']

with open(os.path.join(PROJECT_DIR, 'splits', 'mitbih_patient_split.json')) as f:
    standard_split = json.load(f)
train_records_std = standard_split['train']
val_records_std   = standard_split['val']
test_records_std  = standard_split['test']

print('Standard split (before fix):')
print('  201 in:', 'train' if '201' in train_records_std else ('val' if '201' in val_records_std else 'test'))
print('  202 in:', 'train' if '202' in train_records_std else ('val' if '202' in val_records_std else 'test'))

# ---- STRICT fix: move 202 from test to train, so 201 & 202 (same subject) are together ----
assert '201' in train_records_std, 'Expected 201 in train'
assert '202' in test_records_std, 'Expected 202 in test'

train_records_strict = sorted(train_records_std + ['202'])
val_records_strict   = sorted(val_records_std)          # unchanged
test_records_strict  = sorted([r for r in test_records_std if r != '202'])  # remove 202

print('\nStrict split (after fix):')
print(f'  Train: {len(train_records_strict)} patients')
print(f'  Val:   {len(val_records_strict)} patients')
print(f'  Test:  {len(test_records_strict)} patients')
print('  201 & 202 both now in:', 'train' if ('201' in train_records_strict and '202' in train_records_strict) else 'MISMATCH!')

# Save the strict split manifest
strict_split = {
    'train': train_records_strict, 'val': val_records_strict, 'test': test_records_strict,
    'protocol': 'strict_subject_disjoint (202 moved to train to join subject-mate 201)',
    'random_seed': 42
}
with open(os.path.join(PROJECT_DIR, 'splits', 'mitbih_strict_split.json'), 'w') as f:
    json.dump(strict_split, f, indent=2)
print('\nSaved strict split manifest.')

Standard split (before fix):
  201 in: train
  202 in: test

Strict split (after fix):
  Train: 21 patients
  Val:   5 patients
  Test:  22 patients
  201 & 202 both now in: train

Saved strict split manifest.


In [10]:
AAMI_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V',
    'F': 'F',
    '/': 'Q', 'f': 'Q', 'Q': 'Q'
}
CLASS_TO_IDX = {'N':0, 'S':1, 'V':2, 'F':3, 'Q':4}   # SAME encoding as main pipeline

FS = 360
PRE_SAMPLES  = 99
POST_SAMPLES = 160

def bandpass_filter(signal, fs=FS, low=0.5, high=40.0, order=3):
    nyq = fs / 2.0
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, signal)

def extract_beats(record_id, raw_dir=RAW_DIR):
    sig, fields = wfdb.rdsamp(os.path.join(raw_dir, record_id))
    ann = wfdb.rdann(os.path.join(raw_dir, record_id), 'atr')
    lead = sig[:, 0]
    filtered = bandpass_filter(lead)
    n = len(filtered)

    valid_idx = [i for i, sym in enumerate(ann.symbol) if AAMI_MAP.get(sym)]
    samples = ann.sample[valid_idx]
    symbols = [ann.symbol[i] for i in valid_idx]

    X, y, R = [], [], []
    for idx in range(len(samples)):
        s = samples[idx]
        cls = AAMI_MAP[symbols[idx]]
        start, end = s - PRE_SAMPLES, s + POST_SAMPLES
        if start < 0 or end > n:
            continue

        pre_rr  = (samples[idx] - samples[idx-1]) / FS if idx > 0 else np.nan
        post_rr = (samples[idx+1] - samples[idx]) / FS if idx < len(samples)-1 else np.nan
        w0 = max(0, idx - 5)
        local_rrs = np.diff(samples[w0:idx+1]) / FS
        local_avg_rr = local_rrs.mean() if len(local_rrs) > 0 else np.nan
        ratio = (pre_rr / post_rr) if (post_rr and not np.isnan(post_rr) and post_rr != 0) else np.nan

        X.append(filtered[start:end])
        y.append(CLASS_TO_IDX[cls])
        R.append([pre_rr, post_rr, ratio, local_avg_rr])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    R = np.array(R, dtype=np.float32)

    if len(R):
        col_medians = np.nanmedian(R, axis=0)
        nan_rows, nan_cols = np.where(np.isnan(R))
        R[nan_rows, nan_cols] = col_medians[nan_cols]

    return X, y, R

def build_split_arrays(record_list, label):
    Xs, ys, Rs, rec_ids = [], [], [], []
    for rec in record_list:
        X, y, R = extract_beats(rec)
        Xs.append(X); ys.append(y); Rs.append(R)
        rec_ids.extend([rec] * len(y))
        print(f'  {rec}: {len(y)} beats')
    X_all = np.concatenate(Xs, axis=0)
    y_all = np.concatenate(ys, axis=0)
    R_all = np.concatenate(Rs, axis=0)
    print(f'{label} TOTAL: {X_all.shape[0]} beats\n')
    return X_all, y_all, R_all, np.array(rec_ids)

print('Extracting STRICT-split TRAIN beats...')
X_train_s, y_train_s, R_train_s, rec_train_s = build_split_arrays(train_records_strict, 'TRAIN')
print('Extracting STRICT-split VAL beats...')
X_val_s, y_val_s, R_val_s, rec_val_s = build_split_arrays(val_records_strict, 'VAL')
print('Extracting STRICT-split TEST beats...')
X_test_s, y_test_s, R_test_s, rec_test_s = build_split_arrays(test_records_strict, 'TEST')

# Save with a CLEARLY DISTINCT filename (avoid clashing with any prior "v2" files)
np.savez_compressed(os.path.join(PROCESSED_DIR, 'train_strict.npz'), X=X_train_s, y=y_train_s, R=R_train_s, record_id=rec_train_s)
np.savez_compressed(os.path.join(PROCESSED_DIR, 'val_strict.npz'),   X=X_val_s,   y=y_val_s,   R=R_val_s,   record_id=rec_val_s)
np.savez_compressed(os.path.join(PROCESSED_DIR, 'test_strict.npz'),  X=X_test_s,  y=y_test_s,  R=R_test_s,  record_id=rec_test_s)

from collections import Counter
print('Saved. Class distributions:')
print('  Train:', Counter(y_train_s))
print('  Val:  ', Counter(y_val_s))
print('  Test: ', Counter(y_test_s))

Extracting STRICT-split TRAIN beats...
  101: 1864 beats
  102: 2186 beats
  104: 2227 beats
  106: 2027 beats
  108: 1762 beats
  109: 2531 beats
  112: 2538 beats
  114: 1879 beats
  115: 1952 beats
  116: 2411 beats
  118: 2277 beats
  119: 1987 beats
  124: 1618 beats
  201: 1963 beats
  202: 2135 beats
  203: 2980 beats
  205: 2655 beats
  207: 1859 beats
  215: 3362 beats
  220: 2046 beats
  223: 2604 beats
TRAIN TOTAL: 46863 beats

Extracting STRICT-split VAL beats...
  107: 2136 beats
  122: 2474 beats
  208: 2953 beats
  209: 3004 beats
  230: 2255 beats
VAL TOTAL: 12822 beats

Extracting STRICT-split TEST beats...
  100: 2271 beats
  103: 2083 beats
  105: 2572 beats
  111: 2124 beats
  113: 1794 beats
  117: 1534 beats
  121: 1862 beats
  123: 1517 beats
  200: 2600 beats
  210: 2648 beats
  212: 2747 beats
  213: 3249 beats
  214: 2260 beats
  217: 2208 beats
  219: 2154 beats
  221: 2427 beats
  222: 2481 beats
  228: 2053 beats
  231: 1570 beats
  232: 1780 beats
  233: 3

In [11]:
import tensorflow as tf
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_class_weight

CLASS_NAMES = ['N', 'S', 'V', 'Q']

def drop_F_and_remap(X, y, R, rec):
    mask = y != 3
    X2, y2, R2, rec2 = X[mask], y[mask].copy(), R[mask], rec[mask]
    y2[y2 == 4] = 3
    return X2, y2, R2, rec2

X_train_s, y_train_s, R_train_s, rec_train_s = drop_F_and_remap(X_train_s, y_train_s, R_train_s, rec_train_s)
X_val_s,   y_val_s,   R_val_s,   rec_val_s   = drop_F_and_remap(X_val_s,   y_val_s,   R_val_s,   rec_val_s)
X_test_s,  y_test_s,  R_test_s,  rec_test_s  = drop_F_and_remap(X_test_s,  y_test_s,  R_test_s,  rec_test_s)

print('After F-drop + remap:')
print('  Train:', Counter(y_train_s))
print('  Val:  ', Counter(y_val_s))
print('  Test: ', Counter(y_test_s))

def per_beat_normalize(X, eps=1e-8):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    return (X - m) / (s + eps)

def add_channel(X):
    return X[..., np.newaxis].astype(np.float32)

BATCH_SIZE = 128
def make_ds(X, R, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices(({'waveform': X, 'rr_features': R}, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=42)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Pool train+val (26 patients) for final training; RR stats fit on this pool only
X_pool_s = np.concatenate([X_train_s, X_val_s], axis=0)
y_pool_s = np.concatenate([y_train_s, y_val_s], axis=0)
R_pool_s = np.concatenate([R_train_s, R_val_s], axis=0)

X_pool_s_n = per_beat_normalize(X_pool_s)
X_test_s_n = per_beat_normalize(X_test_s)

pool_rr_mean_s = R_pool_s.mean(axis=0)
pool_rr_std_s  = R_pool_s.std(axis=0) + 1e-8
R_pool_s_n = (R_pool_s - pool_rr_mean_s) / pool_rr_std_s
R_test_s_n = (R_test_s - pool_rr_mean_s) / pool_rr_std_s

# Capped SMOTE (25%, same as main pipeline)
combined_pool_s = np.concatenate([X_pool_s_n, R_pool_s_n], axis=1)
counts = Counter(y_pool_s)
majority = max(counts.values())
strat = {}
for cls, cnt in counts.items():
    if cnt < majority:
        strat[cls] = max(min(int(majority * 0.25), majority), cnt)
min_cnt = min(counts.values())
k = min(5, max(1, min_cnt - 1))
sm = SMOTE(random_state=42, k_neighbors=k, sampling_strategy=strat)
combined_res_s, y_pool_res_s = sm.fit_resample(combined_pool_s, y_pool_s)

beat_len_s = X_pool_s_n.shape[1]
X_pool_res_s = combined_res_s[:, :beat_len_s]
R_pool_res_s = combined_res_s[:, beat_len_s:]

train_ds_s = make_ds(add_channel(X_pool_res_s), R_pool_res_s.astype(np.float32),
                      y_pool_res_s.astype(np.int64), training=True)

# LOCKED config from TransCovNet's CV winner -- NOT re-tuned here
best_config_tc = {'N': 1.0, 'S': 1.5, 'V': 0.8, 'Q': 0.8}
classes_present_s = np.unique(y_pool_res_s)
base_w_s = compute_class_weight('balanced', classes=classes_present_s, y=y_pool_res_s)
base_w_dict_s = dict(zip(classes_present_s.tolist(), base_w_s.tolist()))
final_cw_s = {i: base_w_dict_s[i] * best_config_tc[CLASS_NAMES[i]] for i in classes_present_s}
print('\nFinal class weights (strict split, frozen config):', final_cw_s)

final_model_strict = build_ecg_transcovnet(beat_len_s, seed=42)
final_model_strict.fit(train_ds_s, epochs=30,
                        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=6, restore_best_weights=True)],
                        class_weight=final_cw_s, verbose=1)

final_model_strict.save(os.path.join(PROJECT_DIR, 'models', 'ecg_transcovnet_strict_v2_locked.keras'))

# ===== ONE-TIME LOCKED EVALUATION on strict test set =====
test_inputs_s = {'waveform': add_channel(X_test_s_n), 'rr_features': R_test_s_n.astype(np.float32)}
probs_strict = final_model_strict.predict(test_inputs_s, verbose=0)
preds_strict = np.argmax(probs_strict, axis=1)

print('\n===== 🔒 LOCKED TEST RESULTS — ECG-TransCovNet (STRICT, clean rebuild) =====\n')
print(classification_report(y_test_s, preds_strict, target_names=CLASS_NAMES, digits=4, zero_division=0))

macro_f1_strict = f1_score(y_test_s, preds_strict, average='macro')
weighted_f1_strict = f1_score(y_test_s, preds_strict, average='weighted')
acc_strict = (preds_strict == y_test_s).mean()
print(f'Accuracy: {acc_strict:.4f} | Macro-F1: {macro_f1_strict:.4f} | Weighted-F1: {weighted_f1_strict:.4f}')

cm_strict = confusion_matrix(y_test_s, preds_strict)
print('\nConfusion matrix:')
print('     ', '  '.join(f'{c:>5s}' for c in CLASS_NAMES))
for i, row in enumerate(cm_strict):
    print(f'{CLASS_NAMES[i]:>5s}', '  '.join(f'{v:5d}' for v in row))

with open(os.path.join(PROJECT_DIR, 'outputs', 'strict_split_locked_results_v3.json'), 'w') as f:
    json.dump({'protocol': 'strict_subject_disjoint_clean_rebuild',
               'accuracy': float(acc_strict), 'macro_f1': float(macro_f1_strict),
               'weighted_f1': float(weighted_f1_strict)}, f, indent=2)
print('\n✅ Clean strict-split result saved.')

After F-drop + remap:
  Train: Counter({np.int64(0): 39236, np.int64(3): 4151, np.int64(2): 2819, np.int64(1): 614})
  Val:   Counter({np.int64(0): 8933, np.int64(3): 2079, np.int64(2): 1053, np.int64(1): 385})
  Test:  Counter({np.int64(0): 42423, np.int64(2): 3363, np.int64(3): 1809, np.int64(1): 1782})

Final class weights (strict split, frozen config): {np.int64(0): 0.43749610745500217, np.int64(1): 2.625031141006477, np.int64(2): 1.400016608536788, np.int64(3): 1.400016608536788}
Epoch 1/30
659/659 ━━━━━━━━━━━━━━━━━━━━ 109s 123ms/step - accuracy: 0.8660 - loss: 0.4354
Epoch 2/30
659/659 ━━━━━━━━━━━━━━━━━━━━ 63s 95ms/step - accuracy: 0.9474 - loss: 0.1968
Epoch 3/30
659/659 ━━━━━━━━━━━━━━━━━━━━ 59s 90ms/step - accuracy: 0.9611 - loss: 0.1439
Epoch 4/30
659/659 ━━━━━━━━━━━━━━━━━━━━ 61s 92ms/step - accuracy: 0.9668 - loss: 0.1203
Epoch 5/30
659/659 ━━━━━━━━━━━━━━━━━━━━ 61s 92ms/step - accuracy: 0.9693 - loss: 0.1104
Epoch 6/30
659/659 ━━━━━━━━━━━━━━━━━━━━ 60s 91ms/step - accuracy: 0.